# Clase 15 · Notebook 01
# Una CNN completa y sus mapas de características

**Introducción al Deep Learning — Módulo 3, Unidad 1: Redes convolucionales (Parte II)**

Ensamblamos las piezas de la Parte I en una **CNN completa**, la entrenamos para clasificar dígitos y, lo
más interesante, **miramos qué ve la red por dentro**: los mapas de características que aprende cada capa
convolucional.

1. Construir la CNN (extracción + clasificación).
2. Entrenar y evaluar (accuracy + matriz de confusión).
3. Visualizar los **mapas de características**.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Datos y CNN

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
tf.random.set_seed(42); np.random.seed(42)

digits = load_digits()
X = digits.images.reshape(-1, 8, 8, 1).astype("float32") / 16.0
y = tf.keras.utils.to_categorical(digits.target, 10)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=digits.target)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input

modelo = Sequential([
    Input(shape=(8, 8, 1)),
    Conv2D(8,  (3, 3), padding="same", activation="relu", name="conv1"),
    MaxPooling2D((2, 2)),
    Conv2D(16, (3, 3), padding="same", activation="relu", name="conv2"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(32, activation="relu"),
    Dense(10, activation="softmax"),
])
modelo.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
modelo.summary()

## 2. Entrenar y evaluar

In [ ]:
historia = modelo.fit(Xtr, ytr, epochs=20, batch_size=32, validation_split=0.1, verbose=0)
print("Accuracy en test: %.1f %%" % (modelo.evaluate(Xte, yte, verbose=0)[1] * 100))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
y_pred = modelo.predict(Xte, verbose=0).argmax(axis=1)
y_true = yte.argmax(axis=1)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred)).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusión (10 dígitos)"); plt.tight_layout(); plt.show()

## 3. ¿Qué ve la red? Los mapas de características

Creamos un modelo auxiliar que devuelve las salidas de las capas convolucionales y las dibujamos para una
imagen concreta. Cada mapa resalta un rasgo distinto que la red ha aprendido a detectar.

In [ ]:
from tensorflow.keras.models import Model

ejemplo = Xte[0:1]
plt.figure(figsize=(2, 2)); plt.imshow(ejemplo[0].reshape(8, 8), cmap="gray")
plt.title("Imagen de entrada"); plt.axis("off"); plt.show()

# Modelo que expone las salidas de conv1 y conv2
extractor = Model(inputs=modelo.inputs,
                  outputs=[modelo.get_layer("conv1").output, modelo.get_layer("conv2").output])
mapa1, mapa2 = extractor.predict(ejemplo, verbose=0)

for nombre, mapa in [("conv1 (8 mapas)", mapa1), ("conv2 (16 mapas)", mapa2)]:
    n = mapa.shape[-1]
    fig, axes = plt.subplots(1, n, figsize=(n * 0.9, 1.2))
    for i, ax in enumerate(axes):
        ax.imshow(mapa[0, :, :, i], cmap="viridis"); ax.axis("off")
    fig.suptitle(nombre, y=1.3); plt.show()

Cada recuadro es un **mapa de características**: la respuesta de un filtro aprendido. Los primeros
detectan cosas simples (bordes); los siguientes, combinaciones más complejas. Así es como una CNN
"entiende" una imagen.

## Resumen

- Una **CNN completa** encadena extracción (Conv + Pooling + Flatten) y clasificación (Dense + Softmax).
- Se entrena y evalúa como cualquier clasificador (accuracy, matriz de confusión).
- Los **mapas de características** muestran qué detecta cada capa: la red aprende sus propios filtros.

➡️ En el **Notebook 02** ampliaremos los datos con **aumentación** y añadiremos **batch normalization**.